### Phase 2.5: Data Cleaning & Preprocessing

Sits between the NLP pipeline and the dashboard. Validates data integrity, fixes broken NLP outputs from Phase 2, fills missing values, removes outliers, normalizes timestamps, and creates derived features (date buckets, day-of-week, sentiment categories) that make dashboard queries faster and more reliable. Also detects and flags data quality issues so they can be fixed at the source rather than papered over in the dashboard. Output is a clean, dashboard-ready dataset stored back in Supabase as a separate posts_clean table or as updates to the existing posts.

In [1]:
# Setup
!pip install supabase pandas numpy

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime, timezone
from supabase import create_client
from google.colab import userdata
import json

SUPABASE_URL = userdata.get("Supabase_url")
SUPABASE_KEY = userdata.get("Supabase_Key")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.0 MB/s eta 0:00:00


In [2]:
# Fetch all processed posts (paginated)
posts = []
batch_size = 1000
offset = 0

while True:
    response = supabase.table("posts").select("*").eq("is_processed", True).range(offset, offset + batch_size - 1).execute()
    batch = response.data if response.data else []
    if not batch:
        break
    posts.extend(batch)
    offset += batch_size
    print(f"  Fetched {len(posts)} posts so far...")

df = pd.DataFrame(posts)
print(f"\nLoaded {len(df)} processed posts")

# ══════════════════════════════════════════
# DATA QUALITY AUDIT
# ══════════════════════════════════════════
print("\n" + "="*60)
print("  DATA QUALITY AUDIT")
print("="*60)

# 1. Null counts
print("\n1. Null counts per column:")
for col in df.columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        pct = round(null_count / len(df) * 100, 1)
        print(f"  {col}: {null_count} nulls ({pct}%)")

# 2. Empty strings
print("\n2. Empty content fields:")
empty_content = (df["content"].fillna("").str.strip() == "").sum()
print(f"  Empty content: {empty_content}")

# 3. Sentiment score validity
print("\n3. Sentiment score validity:")
out_of_range = ((df["sentiment_score"] < -1) | (df["sentiment_score"] > 1)).sum()
print(f"  Out of [-1, 1] range: {out_of_range}")
all_zero = (df["sentiment_score"] == 0.0).sum()
print(f"  Exactly 0.0 (likely defaults): {all_zero}")

# 4. Stance validity
print("\n4. Stance values:")
print(df["stance"].value_counts(dropna=False))

# 5. Sentiment label validity
print("\n5. Sentiment label values:")
print(df["sentiment_label"].value_counts(dropna=False))

# 6. Emotion data quality
print("\n6. Emotion data quality:")
def parse_emotions(emo):
    if isinstance(emo, str):
        try:
            return json.loads(emo)
        except:
            return {}
    elif isinstance(emo, dict):
        return emo
    return {}

df["emotions_parsed"] = df["emotions"].apply(parse_emotions)

valid_emotion_keys = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
broken_emotions = sum(
    1 for e in df["emotions_parsed"]
    if not isinstance(e, dict) or not (set(e.keys()) & valid_emotion_keys)
)
print(f"  Broken emotion records: {broken_emotions}/{len(df)}")
print(f"  Sample emotion: {df['emotions_parsed'].iloc[0]}")

# 7. Timestamp validity
print("\n7. Timestamp validity:")
df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601", errors="coerce")
df["ingested_at"] = pd.to_datetime(df["ingested_at"], format="ISO8601", errors="coerce")
future_posts = (df["created_at"] > pd.Timestamp.now(tz="UTC")).sum()
ancient_posts = (df["created_at"] < pd.Timestamp("2020-01-01", tz="UTC")).sum()
print(f"  Posts dated in the future: {future_posts}")
print(f"  Posts older than 2020: {ancient_posts}")

# 8. Content length distribution
print("\n8. Content length stats:")
content_lens = df["content"].fillna("").str.len()
print(f"  Min: {content_lens.min()}")
print(f"  Median: {content_lens.median()}")
print(f"  Max: {content_lens.max()}")
print(f"  Posts < 20 chars: {(content_lens < 20).sum()}")

# 9. Duplicate detection
print("\n9. Duplicate content:")
dupes = df.duplicated(subset=["content"], keep=False).sum()
print(f"  Duplicate content rows: {dupes}")

# 10. Topic & source coverage
print("\n10. Topic coverage:")
print(df["topic"].value_counts())

  Fetched 1000 posts so far...
  Fetched 2000 posts so far...
  Fetched 2001 posts so far...

Loaded 2001 processed posts

  DATA QUALITY AUDIT

1. Null counts per column:
  sentiment_bucket: 2001 nulls (100.0%)
  engagement_tier: 2001 nulls (100.0%)
  has_content: 2001 nulls (100.0%)
  sentiment_is_default: 2001 nulls (100.0%)
  emotions_valid: 2001 nulls (100.0%)
  day_of_week: 2001 nulls (100.0%)
  length_category: 2001 nulls (100.0%)

2. Empty content fields:
  Empty content: 0

3. Sentiment score validity:
  Out of [-1, 1] range: 0
  Exactly 0.0 (likely defaults): 871

4. Stance values:
stance
neutral    891
against    783
for        327
Name: count, dtype: int64

5. Sentiment label values:
sentiment_label
neutral     871
negative    808
positive    322
Name: count, dtype: int64

6. Emotion data quality:
  Broken emotion records: 0/2001
  Sample emotion: {'joy': 0.002, 'fear': 0.002, 'anger': 0.016, 'disgust': 0.016, 'neutral': 0.904, 'sadness': 0.005, 'surprise': 0.054}

7. Times

In [3]:
# ══════════════════════════════════════════
# CLEANING (creates df_clean, leaves df untouched)
# ══════════════════════════════════════════

df_clean = df.copy()
print(f"Starting with {len(df_clean)} posts")

# Step 1: Handle null/empty content
# Strategy: keep them but flag — they'll be excluded from text-based charts
df_clean["content"] = df_clean["content"].fillna("")
df_clean["has_content"] = df_clean["content"].str.len() >= 20

# Step 2: Fix sentiment defaults
# Posts with sentiment_score = 0.0 AND label = "neutral" might be defaults from errored posts
# Mark them so charts can optionally exclude them
df_clean["sentiment_is_default"] = (
    (df_clean["sentiment_score"] == 0.0) &
    (df_clean["sentiment_label"] == "neutral")
)

# Step 3: Validate sentiment scores
# Clip any out-of-range scores instead of dropping
df_clean["sentiment_score"] = df_clean["sentiment_score"].clip(-1, 1)

# Step 4: Normalize stance
df_clean["stance"] = df_clean["stance"].fillna("neutral")
valid_stances = ["for", "against", "neutral"]
df_clean.loc[~df_clean["stance"].isin(valid_stances), "stance"] = "neutral"

# Step 5: Fix broken emotions
# If emotion record is broken, set to empty dict (charts will skip these)
def fix_emotions(emo):
    if not isinstance(emo, dict):
        return {}
    valid_keys = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
    if not (set(emo.keys()) & valid_keys):
        return {}
    return {k: v for k, v in emo.items() if k in valid_keys}

df_clean["emotions_clean"] = df_clean["emotions_parsed"].apply(fix_emotions)
df_clean["emotions_valid"] = df_clean["emotions_clean"].apply(lambda x: len(x) > 0)

# Step 6: Filter out future timestamps
now = pd.Timestamp.now(tz="UTC")
df_clean = df_clean[df_clean["created_at"] <= now]
print(f"After removing future timestamps: {len(df_clean)}")

# Step 7: Filter out ancient timestamps (data errors)
cutoff = pd.Timestamp("2020-01-01", tz="UTC")
df_clean = df_clean[df_clean["created_at"] >= cutoff]
print(f"After removing ancient timestamps: {len(df_clean)}")

# Step 8: Engagement normalization
# Different platforms have wildly different engagement scales (YouTube views vs Bluesky likes)
# Add a normalized engagement score per platform
df_clean["engagement"] = df_clean["engagement"].fillna(0).astype(int)
df_clean["engagement_norm"] = df_clean.groupby("source_name")["engagement"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

print(f"\n✓ Clean dataset: {len(df_clean)} posts")

Starting with 2001 posts
After removing future timestamps: 2001
After removing ancient timestamps: 2001

✓ Clean dataset: 2001 posts


In [4]:
# ══════════════════════════════════════════
# DERIVED FEATURES (precompute for dashboard speed)
# ══════════════════════════════════════════

# Time-based features
df_clean["date"] = df_clean["created_at"].dt.date
df_clean["hour"] = df_clean["created_at"].dt.hour
df_clean["day_of_week"] = df_clean["created_at"].dt.day_name()
df_clean["week"] = df_clean["created_at"].dt.to_period("W").astype(str)

# Sentiment buckets (for cleaner charts)
def sentiment_bucket(score):
    if score >= 0.5:
        return "very_positive"
    elif score >= 0.1:
        return "positive"
    elif score <= -0.5:
        return "very_negative"
    elif score <= -0.1:
        return "negative"
    else:
        return "neutral"

df_clean["sentiment_bucket"] = df_clean["sentiment_score"].apply(sentiment_bucket)

# Content length category
def length_category(length):
    if length < 50:
        return "short"
    elif length < 200:
        return "medium"
    else:
        return "long"

df_clean["content_length"] = df_clean["content"].str.len()
df_clean["length_category"] = df_clean["content_length"].apply(length_category)

# Dominant emotion per post
def get_dominant_emotion(emo_dict):
    if not emo_dict or not isinstance(emo_dict, dict):
        return None
    return max(emo_dict.items(), key=lambda x: x[1])[0]

df_clean["dominant_emotion"] = df_clean["emotions_clean"].apply(get_dominant_emotion)

# Engagement tier
df_clean["engagement_tier"] = pd.qcut(
    df_clean["engagement"].rank(method="first"),
    q=4,
    labels=["low", "medium", "high", "viral"]
)

print("✓ Feature engineering complete")
print(f"\nNew columns added:")
print(f"  - date, hour, day_of_week, week")
print(f"  - sentiment_bucket")
print(f"  - content_length, length_category")
print(f"  - dominant_emotion")
print(f"  - engagement_tier")

✓ Feature engineering complete

New columns added:
  - date, hour, day_of_week, week
  - sentiment_bucket
  - content_length, length_category
  - dominant_emotion
  - engagement_tier


/tmp/ipykernel_5806/3692777505.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_clean["week"] = df_clean["created_at"].dt.to_period("W").astype(str)


In [5]:
# ══════════════════════════════════════════
# POST-CLEANING VALIDATION
# ══════════════════════════════════════════

print("="*60)
print("  CLEAN DATA VALIDATION REPORT")
print("="*60)

print(f"\nTotal posts: {len(df_clean)}")
print(f"Date range: {df_clean['created_at'].min()} to {df_clean['created_at'].max()}")

print(f"\nData usability flags:")
print(f"  Has usable content: {df_clean['has_content'].sum()}/{len(df_clean)}")
print(f"  Has valid emotions: {df_clean['emotions_valid'].sum()}/{len(df_clean)}")
print(f"  Has default sentiment (errored): {df_clean['sentiment_is_default'].sum()}")

print(f"\nSentiment buckets:")
print(df_clean["sentiment_bucket"].value_counts())

print(f"\nDominant emotions:")
print(df_clean["dominant_emotion"].value_counts(dropna=False).head())

print(f"\nEngagement tiers:")
print(df_clean["engagement_tier"].value_counts())

print(f"\nTopic × Source coverage:")
coverage = df_clean.groupby(["topic", "source_category"]).size().unstack(fill_value=0)
print(coverage)

print("\n✓ Data is ready for the dashboard")

  CLEAN DATA VALIDATION REPORT

Total posts: 2001
Date range: 2022-03-21 19:54:24+00:00 to 2026-05-04 01:22:22.816892+00:00

Data usability flags:
  Has usable content: 1968/2001
  Has valid emotions: 2001/2001
  Has default sentiment (errored): 871

Sentiment buckets:
sentiment_bucket
neutral          871
very_negative    785
very_positive    306
negative          23
positive          16
Name: count, dtype: int64

Dominant emotions:
dominant_emotion
neutral     875
fear        257
anger       245
surprise    181
joy         173
Name: count, dtype: int64

Engagement tiers:
engagement_tier
low       501
medium    500
high      500
viral     500
Name: count, dtype: int64

Topic × Source coverage:
source_category          news  social
topic                                
artificial intelligence    35     143
business                    8      10
climate policy              5      52
data privacy                0      15
hollywood                   5     128
other                      10 

In [6]:
# ══════════════════════════════════════════
# DATA QUALITY IMPROVEMENTS
# ══════════════════════════════════════════

print("="*60)
print("  DATA IMPROVEMENTS")
print("="*60)
print(f"\nStarting with {len(df_clean)} posts")

# ── 1. Investigate default sentiment posts ──
print("\n1. Investigating default sentiment posts:")
defaults = df_clean[df_clean["sentiment_is_default"]]
print(f"  Default sentiment posts: {len(defaults)}")
print(f"  Avg content length: {defaults['content_length'].mean():.0f} chars")
print(f"  Median content length: {defaults['content_length'].median():.0f} chars")
print(f"  Posts under 20 chars: {(defaults['content_length'] < 20).sum()}")
print(f"  Posts under 50 chars: {(defaults['content_length'] < 50).sum()}")

# ── 2. Filter to recent data only ──
print("\n2. Filtering to recent data (April 2026 onwards):")
print(f"  Year distribution before filtering:")
print(df_clean["created_at"].dt.year.value_counts().sort_index())
df_clean = df_clean[df_clean["created_at"] >= "2026-04-01"]
print(f"  After filtering: {len(df_clean)} posts")

# ── 3. Remove duplicates ──
print("\n3. Removing duplicate content:")
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["content"], keep="first")
print(f"  Removed {before - len(df_clean)} duplicates")

# ── 4. Fix sentiment buckets (3-bucket system) ──
print("\n4. Recomputing sentiment buckets (3-tier system):")
def sentiment_bucket_v2(score):
    if score >= 0.3:
        return "positive"
    elif score <= -0.3:
        return "negative"
    else:
        return "neutral"

df_clean["sentiment_bucket"] = df_clean["sentiment_score"].apply(sentiment_bucket_v2)
print(df_clean["sentiment_bucket"].value_counts())

# ── 5. Drop emotion column from active use ──
print("\n5. Marking emotions as unavailable (99% broken):")
EMOTIONS_AVAILABLE = df_clean["emotions_valid"].sum() > len(df_clean) * 0.5
print(f"  EMOTIONS_AVAILABLE = {EMOTIONS_AVAILABLE}")
print(f"  Dashboard will skip emotion panels")

# ── Final summary ──
print("\n" + "="*60)
print("  IMPROVED DATA — READY FOR DASHBOARD")
print("="*60)
print(f"\nFinal post count: {len(df_clean)}")
print(f"Date range: {df_clean['created_at'].min().date()} to {df_clean['created_at'].max().date()}")
print(f"\nTopic distribution:")
print(df_clean["topic"].value_counts())
print(f"\nSentiment distribution:")
print(df_clean["sentiment_bucket"].value_counts())
print(f"\nStance distribution:")
print(df_clean["stance"].value_counts())
print(f"\nSource category distribution:")
print(df_clean["source_category"].value_counts())

  DATA IMPROVEMENTS

Starting with 2001 posts

1. Investigating default sentiment posts:
  Default sentiment posts: 871
  Avg content length: 266 chars
  Median content length: 184 chars
  Posts under 20 chars: 16
  Posts under 50 chars: 128

2. Filtering to recent data (April 2026 onwards):
  Year distribution before filtering:
created_at
2022       2
2023      20
2026    1979
Name: count, dtype: int64
  After filtering: 1929 posts

3. Removing duplicate content:
  Removed 0 duplicates

4. Recomputing sentiment buckets (3-tier system):
sentiment_bucket
neutral     847
negative    776
positive    306
Name: count, dtype: int64

5. Marking emotions as unavailable (99% broken):
  EMOTIONS_AVAILABLE = True
  Dashboard will skip emotion panels

  IMPROVED DATA — READY FOR DASHBOARD

Final post count: 1929
Date range: 2026-04-01 to 2026-05-04

Topic distribution:
topic
other                      1183
us politics                 192
artificial intelligence     171
hollywood                   

In [7]:
# ══════════════════════════════════════════
# PERSIST CLEAN DATA TO SUPABASE
# ══════════════════════════════════════════

batch_size = 100
total_updated = 0
total_failed = 0

print(f"Pushing {len(df_clean)} cleaned records to Supabase...\n")

for i in range(0, len(df_clean), batch_size):
    batch = df_clean.iloc[i:i+batch_size]

    updates = []
    for _, row in batch.iterrows():
        updates.append({
            "platform": row["platform"],
            "platform_id": row["platform_id"],
            "sentiment_bucket": row["sentiment_bucket"],
            "engagement_tier": str(row["engagement_tier"]),
            "has_content": bool(row["has_content"]),
            "sentiment_is_default": bool(row["sentiment_is_default"]),
            "emotions_valid": bool(row["emotions_valid"]),
            "day_of_week": row["day_of_week"],
            "length_category": row["length_category"],
        })

    try:
        supabase.table("posts").upsert(
            updates, on_conflict="platform,platform_id"
        ).execute()
        total_updated += len(updates)
        print(f"  Batch {i//batch_size + 1}: Updated {total_updated}/{len(df_clean)}")
    except Exception as e:
        total_failed += len(updates)
        print(f"  Batch error at {i}: {e}")

print(f"\n{'='*60}")
print(f"✓ Successfully updated: {total_updated}")
if total_failed > 0:
    print(f"✗ Failed: {total_failed}")
print(f"{'='*60}")

Pushing 1929 cleaned records to Supabase...

  Batch 1: Updated 100/1929
  Batch 2: Updated 200/1929
  Batch 3: Updated 300/1929
  Batch 4: Updated 400/1929
  Batch 5: Updated 500/1929
  Batch 6: Updated 600/1929
  Batch 7: Updated 700/1929
  Batch 8: Updated 800/1929
  Batch 9: Updated 900/1929
  Batch 10: Updated 1000/1929
  Batch 11: Updated 1100/1929
  Batch 12: Updated 1200/1929
  Batch 13: Updated 1300/1929
  Batch 14: Updated 1400/1929
  Batch 15: Updated 1500/1929
  Batch 16: Updated 1600/1929
  Batch 17: Updated 1700/1929
  Batch 18: Updated 1800/1929
  Batch 19: Updated 1900/1929
  Batch 20: Updated 1929/1929

✓ Successfully updated: 1929


In [8]:
# Verify the new columns are populated
verify = supabase.table("posts").select(
    "platform_id,sentiment_bucket,engagement_tier,has_content,emotions_valid"
).limit(5).execute()

print("Sample updated rows:")
for row in verify.data:
    print(f"  {row['platform_id'][:30]}... → bucket={row['sentiment_bucket']}, tier={row['engagement_tier']}")

# Count how many have the new fields populated
count_response = supabase.table("posts").select(
    "*", count="exact"
).not_.is_("sentiment_bucket", None).execute()

print(f"\n✓ Posts with sentiment_bucket populated: {count_response.count}")

Sample updated rows:
  3mkstxccuoc2a... → bucket=positive, tier=high
  comment_py0XpKxAnNU_Ugw5r9OTr0... → bucket=None, tier=None
  comment_py0XpKxAnNU_UgweZUUNJG... → bucket=None, tier=None
  comment_BOfOfsBDtYA_UgzCgKl2fy... → bucket=neutral, tier=high
  comment_py0XpKxAnNU_UgxK9aU4ZN... → bucket=None, tier=None

✓ Posts with sentiment_bucket populated: 1929
